# Capítulo 15: Consultas en Lenguaje Natural y Copilotos de BI

## Notebook de Práctica

En este notebook exploraremos cómo usar un copiloto de BI basado en lenguaje natural para realizar consultas sobre datos de RRHH. Usaremos:
- Traducción NL→SQL con IA
- Análisis de resultados y visualización
- Dashboard de RRHH interactivo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import json

# Cargar datos
df = pd.read_csv('../datos/datos_rrhh_empleados.csv')
print(f'Dataset cargado: {df.shape[0]} empleados, {df.shape[1]} columnas')
df.head(10)

---
## Celda 1: Exploración Inicial del Dataset

Antes de usar el copiloto, exploremos la estructura de los datos para entender qué podemos preguntar.

In [ ]:
print('=== Información del Dataset ===')
print(f'Empleados totales: {len(df)}')
print(f'\nDepartamentos: {df["department"].nunique()}')
print(df['department'].value_counts().to_string())
print(f'\nRangos de ingresos: $' + f'{df["monthly_income"].min():,.0f}' + ' - $' + f'{df["monthly_income"].max():,.0f}')
print(f'\nDistribución de riesgo de rotación:')
print(df['attrition_risk'].value_counts().to_string())

---
## Celda 2: Configurar Base de Datos SQLite para el Copiloto

El copiloto necesita un esquema de base de datos para generar consultas SQL. Cargaremos los datos en SQLite.

In [ ]:
# Crear base de datos en memoria
conn = sqlite3.connect(':memory:')
df.to_sql('empleados', conn, index=False, if_exists='replace')

# Verificar esquema
cursor = conn.execute("PRAGMA table_info(empleados)")
schema = cursor.fetchall()
print('=== Esquema de la tabla empleados ===')
for col in schema:
    print(f'  {col[1]:25s} ({col[2]})')

# Verificar conteo
count = conn.execute('SELECT COUNT(*) FROM empleados').fetchone()[0]
print(f'\nRegistros cargados: {count}')

---
## Celda 3: Simulación de Copiloto NL→SQL

Simularemos lo que haría un copiloto de BI real: tomar una pregunta en lenguaje natural y generar el SQL correspondiente.

In [ ]:
# Mapa de preguntas NL → SQL (simula lo que haría un LLM)
nl_sql_map = {
    "¿Cuántos empleados tienen riesgo alto?": 
        "SELECT COUNT(*) as total FROM empleados WHERE attrition_risk = 'Alto';",
    
    "¿Cuáles son los 5 departamentos con más rotación?": 
        """SELECT department, 
                  COUNT(CASE WHEN attrition_risk = 'Alto' THEN 1 END) as en_riesgo,
                  ROUND(COUNT(CASE WHEN attrition_risk = 'Alto' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_riesgo
           FROM empleados 
           GROUP BY department 
           ORDER BY pct_riesgo DESC 
           LIMIT 5;""",
    
    "¿Cuál es el ingreso promedio por departamento?": 
        """SELECT department, 
                  ROUND(AVG(monthly_income), 2) as ingreso_promedio,
                  COUNT(*) as empleados
           FROM empleados 
           GROUP BY department 
           ORDER BY ingreso_promedio DESC;""",
    
    "¿Los empleados con horas extras tienen más riesgo?": 
        """SELECT overtime, 
                  attrition_risk, 
                  COUNT(*) as total
           FROM empleados 
           GROUP BY overtime, attrition_risk
           ORDER BY overtime, attrition_risk;""",
    
    "¿Cómo se relaciona educación con salario?": 
        """SELECT education, 
                  ROUND(AVG(monthly_income), 2) as ingreso_promedio,
                  MIN(monthly_income) as minimo,
                  MAX(monthly_income) as maximo
           FROM empleados 
           GROUP BY education
           ORDER BY ingreso_promedio DESC;""",
    
    "¿Qué departamentos tienen mejor satisfacción laboral?": 
        """SELECT department, 
                  ROUND(AVG(job_satisfaction), 2) as satisfaccion_promedio,
                  COUNT(*) as empleados
           FROM empleados 
           GROUP BY department
           ORDER BY satisfaccion_promedio DESC;"""
}

print('=== Preguntas disponibles para el copiloto ===')
for i, pregunta in enumerate(nl_sql_map.keys(), 1):
    print(f'{i}. {pregunta}')

---
## Celda 4: Ejecutar Consultas del Copiloto

Veamos cómo el copiloto traduce y ejecuta cada pregunta.

In [ ]:
def copiloto_bi(pregunta):
    """Simula un copiloto de BI: NL → SQL → Resultado"""
    if pregunta not in nl_sql_map:
        return None, None, 'Pregunta no reconocida. Intenta con otra pregunta.'
    
    sql = nl_sql_map[pregunta]
    try:
        resultado = pd.read_sql_query(sql, conn)
        return pregunta, sql, resultado
    except Exception as e:
        return pregunta, sql, f'Error: {e}'

# Ejecutar todas las preguntas
for pregunta in nl_sql_map.keys():
    _, sql, resultado = copiloto_bi(pregunta)
    print(f'\n{"="*60}')
    print(f'PREGUNTA: {pregunta}')
    print(f'\nSQL generado:')
    print(f'{sql}')
    print(f'\nResultado:')
    if isinstance(resultado, pd.DataFrame):
        print(resultado.to_string(index=False))
    else:
        print(resultado)

---
## Celda 5: Visualización de Resultados del Copiloto

El copiloto no solo genera datos: también puede sugerir visualizaciones.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Dashboard de RRHH - Generado por Copiloto BI', fontsize=16, fontweight='bold')

# 1. Riesgo de rotación por departamento
rotacion = df.groupby('department').apply(
    lambda x: (x['attrition_risk'] == 'Alto').sum() / len(x) * 100
).sort_values(ascending=True)
rotacion.plot(kind='barh', ax=axes[0, 0], color='coral')
axes[0, 0].set_title('% Empleados con Riesgo Alto por Departamento')
axes[0, 0].set_xlabel('% de riesgo')

# 2. Ingreso promedio por departamento
ingreso_dept = df.groupby('department')['monthly_income'].mean().sort_values(ascending=True)
ingreso_dept.plot(kind='barh', ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Ingreso Promedio por Departamento')
axes[0, 1].set_xlabel('Ingreso mensual ($)')

# 3. Satisfacción laboral por horas extras
satis_overtime = df.groupby(['overtime', 'attrition_risk']).size().unstack(fill_value=0)
satis_overtime.plot(kind='bar', ax=axes[1, 0], colormap='RdYlGn_r')
axes[1, 0].set_title('Horas Extras vs Riesgo de Rotación')
axes[1, 0].set_xlabel('Horas Extras')
axes[1, 0].set_ylabel('Cantidad de empleados')
axes[1, 0].tick_params(axis='x', rotation=0)

# 4. Educación vs Salario
edu_sal = df.groupby('education')['monthly_income'].agg(['mean', 'std']).reindex(
    ['Sin título formal', 'Secundaria', 'Licenciatura', 'Maestría', 'Doctorado']
)
edu_sal['mean'].plot(kind='bar', ax=axes[1, 1], color='mediumseagreen', 
                     yerr=edu_sal['std'], capsize=5)
axes[1, 1].set_title('Nivel Educativo vs Ingreso Promedio')
axes[1, 1].set_xlabel('Nivel Educativo')
axes[1, 1].set_ylabel('Ingreso mensual ($)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../datos/dashboard_rrhh_cap15.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard guardado en datos/dashboard_rrhh_cap15.png')

---
## Celda 6: Análisis de Correlación NL→SQL con Prompt Engineering

Compara la calidad de las traducciones usando diferentes técnicas de prompt.

In [ ]:
# Ejemplo de cómo un LLM generaría SQL con diferentes enfoques
prompt_basico = """Pregunta: ¿Cuántos empleados de riesgo alto hay en ventas?

SQL generado:
SELECT COUNT(*) FROM empleados 
WHERE department = 'Ventas' AND attrition_risk = 'Alto';"""

prompt_detallado = """Eres un experto SQL. Esquema: empleados(employee_id, department, role, 
monthly_income, years_at_company, job_satisfaction, overtime, attrition_risk, 
performance_rating, education)

Pregunta: ¿Cuántos empleados de riesgo alto hay en ventas?

SQL generado:
SELECT 
    department,
    COUNT(*) as empleados_en_riesgo,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM empleados WHERE department = 'Ventas'), 2) as porcentaje
FROM empleados
WHERE department = 'Ventas' AND attrition_risk = 'Alto'
GROUP BY department;"""

print('=== Comparación de Enfoques de Prompt ===')
print('\nPrompt Básico:')
print(prompt_basico)
print('\n' + '='*50)
print('\nPrompt Detallado (con contexto de esquema):')
print(prompt_detallado)

# Ejecutar ambas consultas
sql_basico = "SELECT COUNT(*) as total FROM empleados WHERE department = 'Ventas' AND attrition_risk = 'Alto'"
sql_detallado = """SELECT department, COUNT(*) as empleados_en_riesgo,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM empleados WHERE department = 'Ventas'), 2) as porcentaje
FROM empleados WHERE department = 'Ventas' AND attrition_risk = 'Alto' GROUP BY department"""

print('\n=== Resultados ===')
print(f'Consulta básica: {pd.read_sql_query(sql_basico, conn).to_dict("records")[0]}')
print(f'Consulta detallada: {pd.read_sql_query(sql_detallado, conn).to_dict("records")[0]}')

---
## Celda 7: Análisis Iterativo del Copiloto

Un copiloto real permite iterar: refinar preguntas y drill-down en los datos.

In [ ]:
# Simular una conversación iterativa con el copiloto
conversacion = [
    {
        'nivel': 1,
        'pregunta': '¿Qué departamentos tienen más riesgo de rotación?',
        'sql': """SELECT department, 
                    COUNT(CASE WHEN attrition_risk = 'Alto' THEN 1 END) as en_riesgo
             FROM empleados GROUP BY department ORDER BY en_riesgo DESC"""
    },
    {
        'nivel': 2,
        'pregunta': '¿Y cuál es la satisfacción promedio de esos departamentos?',
        'sql': """SELECT department, 
                    ROUND(AVG(job_satisfaction), 2) as satisfaccion,
                    COUNT(CASE WHEN attrition_risk = 'Alto' THEN 1 END) as en_riesgo
             FROM empleados 
             WHERE department IN ('Soporte', 'Ventas', 'Operations')
             GROUP BY department"""
    },
    {
        'nivel': 3,
        'pregunta': '¿Qué roles específicos de Soporte tienen mayor riesgo?',
        'sql': """SELECT role, 
                    COUNT(*) as total,
                    COUNT(CASE WHEN attrition_risk = 'Alto' THEN 1 END) as en_riesgo,
                    ROUND(AVG(monthly_income), 2) as ingreso_prom
             FROM empleados 
             WHERE department = 'Soporte'
             GROUP BY role 
             ORDER BY en_riesgo DESC"""
    }
]

print('=== Conversación Iterativa con el Copiloto ===\n')
for paso in conversacion:
    print(f'--- Nivel {paso["nivel"]} ---')
    print(f'PREGUNTA: {paso["pregunta"]}')
    print(f'SQL: {paso["sql"]}')
    resultado = pd.read_sql_query(paso['sql'], conn)
    print(f'RESULTADO:')
    print(resultado.to_string(index=False))
    print()

---
## Celda 8: Evaluación de Precisión del Copiloto

Evalúa qué tan bien el copiloto traduce preguntas en luguaje natural a SQL correcto.

In [ ]:
# Evaluación de traducciones NL→SQL
evaluaciones = [
    {
        'pregunta': '¿Cuántos empleados hay en total?',
        'sql_esperado': 'SELECT COUNT(*) FROM empleados',
        'resultado_esperado': 1500
    },
    {
        'pregunta': '¿Cuál es el ingreso promedio general?',
        'sql_esperado': 'SELECT ROUND(AVG(monthly_income), 2) FROM empleados',
        'resultado_esperado': None  # Verificaremos que sea un número
    },
    {
        'pregunta': '¿Cuántos departamentos hay?',
        'sql_esperado': 'SELECT COUNT(DISTINCT department) FROM empleados',
        'resultado_esperado': 10
    },
    {
        'pregunta': '¿Cuántos empleados tienen doctorado?',
        'sql_esperado': "SELECT COUNT(*) FROM empleados WHERE education = 'Doctorado'",
        'resultado_esperado': None
    },
]

print('=== Evaluación de Precisión NL→SQL ===\n')
aciertos = 0
for eval in evaluaciones:
    resultado = pd.read_sql_query(eval['sql_esperado'], conn)
    valor = resultado.iloc[0, 0]
    
    if eval['resultado_esperado'] is None:
        pass  # Solo verificamos que sea numérico
    elif valor == eval['resultado_esperado']:
        aciertos += 1
    
    print(f'Pregunta: {eval["pregunta"]}')
    print(f'SQL: {eval["sql_esperado"]}')
    print(f'Resultado: {valor}')
    print(f'Estado: {"✓" if eval["resultado_esperado"] is None or valor == eval["resultado_esperado"] else "✗"}')
    print()

print(f'\nTasa de éxito: {aciertos}/{len(evaluaciones)} = {aciertos/len(evaluaciones)*100:.0f}%')

---
## Celda 9: Generación de Reporte Automático

El copiloto puede generar reportes ejecutivos a partir de los datos.

In [ ]:
# Generar reporte ejecutivo de RRHH
total = len(df)
alto = (df['attrition_risk'] == 'Alto').sum()
medio = (df['attrition_risk'] == 'Medio').sum()
bajo = (df['attrition_risk'] == 'Bajo').sum()

reporte = f"""
{'='*60}
REPORTE EJECUTIVO DE RRHH - Generado por Copiloto BI
{'='*60}

RESUMEN EJECUTIVO:
La empresa cuenta con {total} empleados. El {alto/total*100:.1f}% presenta
riesgo alto de rotación, lo que requiere atención inmediata.

MÉTRICAS CLAVE:
  • Empleados totales: {total}
  • Riesgo Alto: {alto} ({alto/total*100:.1f}%)
  • Riesgo Medio: {medio} ({medio/total*100:.1f}%)
  • Riesgo Bajo: {bajo} ({bajo/total*100:.1f}%)
  • Ingreso promedio: ${df['monthly_income'].mean():,.2f}
  • Satisfacción promedio: {df['job_satisfaction'].mean():.2f}/5

DEPARTAMENTOS CON MAYOR RIESGO:
"""

dept_riesgo = df.groupby('department').apply(
    lambda x: (x['attrition_risk'] == 'Alto').sum() / len(x) * 100
).sort_values(ascending=False)

for dept, pct in dept_riesgo.head(5).items():
    reporte += f"  • {dept}: {pct:.1f}% en riesgo alto\n"

reporte += f"""

ANÁLISIS DE HORAS EXTRAS:
  Con horas extras - Riesgo Alto: {(df[df['overtime']=='Sí']['attrition_risk']=='Alto').sum()}
  Sin horas extras - Riesgo Alto: {(df[df['overtime']=='No']['attrition_risk']=='Alto').sum()}

RECOMENDACIONES:
  1. Investigar causas de rotación en departamentos críticos
  2. Revisar política de horas extras y su impacto en satisfacción
  3. Implementar programas de retención para empleados de alto rendimiento

Próximos pasos: Monitorear evolución mensual de estos indicadores.
{'='*60}
"""

print(reporte)

---
## Resumen del Capítulo 15

En este notebook hemos demostrado:

1. **Carga y exploración** de datos de RRHH
2. **Configuración** de una base de datos SQLite para el copiloto
3. **Traducción NL→SQL** con diferentes niveles de complejidad
4. **Visualización automática** de resultados
5. **Análisis iterativo** (drill-down) de los datos
6. **Evaluación de precisión** de las traducciones
7. **Generación de reportes** ejecutivos automatizados

### Lecciones clave:
- Los copilotos de BI democratizan el acceso a los datos
- El contexto del esquema mejora la precisión de las traducciones
- La iteración conversacional permite refinar análisis complejos
- Siempre se debe validar el SQL generado antes de ejecutarlo en producción

### Referencias:
- Li et al. (2024) - Survey on NL to SQL Translation
- Katsogiannis-Meimarakis (2023) - NLIDB Systems Survey